# G2 — Live demo: one head, five encoders, real images

This is the G1 result made visible. It loads the same cached spaces,
builds the same 512-d hub, trains the caption head on **DINOv2-small
only** — and then shows, for real COCO images you can look at, the same
head retrieving captions through encoders it has never seen.

Nothing here is simulated: the vectors are the cached ones the measured
numbers came from, and the captions and images are re-derived from the
COCO annotations by the same deterministic recipe E1 used (captions
grouped per image, ids sorted, prefix taken) — so row *i* of every cache
is image `ids[i]`, and what you see on screen is what the numbers were
measured on.

In [ ]:
import os
from pathlib import Path
STORAGE   = "drive"                     # "drive" | "local" | "env"
DRIVE_DIR = "/content/drive/MyDrive/convergence_experiment"
LOCAL_DIR = "./convergence_data"
try:
    import google.colab                 # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if STORAGE == "env":
    assert os.environ.get("DATA_DIR"), "STORAGE='env' but DATA_DIR unset"
elif STORAGE == "drive" and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["DATA_DIR"] = DRIVE_DIR
else:
    if STORAGE == "drive":
        print("not on Colab - using LOCAL_DIR")
    os.environ["DATA_DIR"] = str(Path(LOCAL_DIR).resolve())
DATA_DIR = Path(os.environ["DATA_DIR"])
print("DATA_DIR:", DATA_DIR)

In [ ]:
# ---- Drive-resilient loading ---------------------------------------
# Colab's Drive mount can drop mid-read on large files ("Transport
# endpoint is not connected"). Two defences: copy the caches to local
# SSD once (immune to the FUSE drop), and wrap every load in a retry
# that remounts before giving up. Local disk also makes the demo faster.
import numpy as np, shutil, time

def _remount():
    try:
        from google.colab import drive
        drive.flush_and_unmount()
        drive.mount("/content/drive", force_remount=True)
        return True
    except Exception:
        return False   # not on Colab, or remount unavailable - just retry

# Read straight from Google Drive by default. The retry-and-remount in
# load_npz() below already handles the occasional mount drop. If Drive
# is dropping repeatedly this session, set this True to copy the caches
# to local SSD once (faster, and immune to the FUSE drop).
USE_LOCAL_COPY = False
if USE_LOCAL_COPY and str(DATA_DIR).startswith("/content/drive"):
    _local = Path("/content/cache"); _local.mkdir(exist_ok=True)
    for _f in sorted(DATA_DIR.glob("*.npz")) + \
              sorted(DATA_DIR.glob("annotations_trainval2017.zip")):
        _dst = _local / _f.name
        if not _dst.exists():
            for _try in range(3):
                try:
                    shutil.copy(str(_f), str(_dst)); break
                except OSError:
                    print(f"  copy retry for {_f.name}...")
                    _remount(); time.sleep(2)
    DATA_DIR = _local
    print("reading caches from local SSD:", DATA_DIR)

def load_npz(path, tries=3):
    """np.load with a remount-and-retry on transient mount drops."""
    path = Path(path)
    for k in range(tries):
        try:
            d = np.load(str(path))
            _ = d.files          # force the central-directory read now
            return d
        except OSError as e:
            print(f"  load failed ({e.errno}) on {path.name}, "
                  f"retry {k+1}/{tries}")
            _remount(); time.sleep(2)
    raise OSError(f"could not read {path} after {tries} tries - "
                  f"remount Drive manually and re-run")

In [ ]:
import numpy as np, json, zipfile

# ---- vectors: same caches G1 used ----
SPACES = {}
for size in ("small", "base", "large"):
    f = DATA_DIR / f"e1_img_ckpt_dinov2-{size}_cls+patch.npz"
    if f.exists():
        SPACES[f"img_{size}"] = load_npz(f)["img"].astype(np.float64)
d = load_npz(DATA_DIR / "crossmodal_pairs.npz")
SPACES["txt_bge"] = d["txt"].astype(np.float64)
# alignment proof, as in G1: the pairs file's image block must equal a cache
_ok = False
for ref in [k for k in SPACES if k.startswith("img_")]:
    if d["img"].shape[1] == SPACES[ref].shape[1]:
        n = min(len(d["img"]), len(SPACES[ref]))
        if np.allclose(d["img"][:n], SPACES[ref][:n], atol=1e-4):
            _ok = True
assert _ok, "pairs file aligns with no image cache - stop"
N = min(len(v) for v in SPACES.values())
SPACES = {k: v[:N] for k, v in SPACES.items()}
print({k: v.shape for k, v in SPACES.items()})

# ---- captions + images: re-derive ids by E1's exact recipe ----
zf = DATA_DIR / "annotations_trainval2017.zip"
assert zf.exists(), "annotations zip not on Drive - run E1/B1 once first"
for _try in range(3):
    try:
        with zipfile.ZipFile(str(zf)) as z:
            ann = json.load(z.open("annotations/captions_train2017.json"))
        break
    except OSError:
        print("  annotations read retry..."); _remount(); time.sleep(2)
url  = {im["id"]: im["coco_url"] for im in ann["images"]}
caps = {}
for a in ann["annotations"]:
    caps.setdefault(a["image_id"], []).append(a["caption"].strip())
ids = sorted(set(caps) & set(url))[:N]
assert len(ids) == N, "id derivation shorter than the caches - stop"
print(f"{N} rows; row i of every cache = image ids[i]")

rng = np.random.default_rng(0)                # SAME split as G1
perm = rng.permutation(N)
te, tr = perm[:1000], perm[1000:]

In [ ]:
# ---- the hub and the single head, exactly as measured ----
HUB_DIM, ALPHA = 512, 1e-2
def l2n(V): return V / (np.linalg.norm(V, axis=-1, keepdims=True) + 1e-12)
def ridge(X, Y, a=ALPHA):
    return np.linalg.solve(X.T @ X + a * np.eye(X.shape[1]), X.T @ Y)

_ref = np.hstack([(SPACES[k][tr] - SPACES[k][tr].mean(0)) /
                  (SPACES[k][tr].std(0).mean() + 1e-12) for k in SPACES])
_mu = _ref.mean(0)
_U, _sv, _VT = np.linalg.svd(_ref - _mu, full_matrices=False)
BASIS = _VT[:HUB_DIM].T / (_sv[:HUB_DIM] / np.sqrt(len(_ref)))
HUB_TR = (_ref - _mu) @ BASIS
TO_HUB = {k: ridge(v[tr], HUB_TR) for k, v in SPACES.items()}

TRAIN_ON = "img_small"

# Which space the head reads OUT into. bge-m3 is the default because
# every reported number uses it, so results stay comparable. SBERT is a
# legitimate alternative - also contrastively trained, and it scored
# slightly higher raw in E1.3 (R@1 0.510 vs 0.466). BERT would work but
# is weaker (0.362, effective rank 65). The hub itself is unaffected by
# this choice: it is target-free, and only the head depends on it.
READOUT = "txt_bge"
assert READOUT in SPACES, f"{READOUT} not loaded; available: {list(SPACES)}"
if READOUT != "txt_bge":
    print(f"NOTE: readout is {READOUT}, not txt_bge - R@1 here is NOT")
    print("comparable with the numbers reported in the project.")
HEAD = ridge(SPACES[TRAIN_ON][tr] @ TO_HUB[TRAIN_ON],
             SPACES[READOUT][tr])
GAL = l2n(SPACES[READOUT][te])              # 1000-caption gallery

def recall_at_1(enc):
    P = l2n((SPACES[enc][te] @ TO_HUB[enc]) @ HEAD)
    return float(((P @ GAL.T).argmax(1) == np.arange(len(te))).mean())

donors = [k for k in SPACES if k.startswith("img_")]
print(f"head trained ONLY on {TRAIN_ON}; sanity vs the measured table:")
for e in donors:
    print(f"   {e:10s} R@1 {recall_at_1(e):.3f}"
          + ("   <- trained here" if e == TRAIN_ON else "   <- never seen"))

## The demo

Each example shows a real held-out COCO image, then the top-3 captions
retrieved by the **same head** through each encoder — including the two
it never saw. A check mark means the true caption of that image ranked
first among all 1,000 candidates.

In [ ]:
import urllib.request
from io import BytesIO
try:
    from PIL import Image as PILImage
    from IPython.display import display
    _CAN_SHOW = True
except ImportError:
    _CAN_SHOW = False

def show_examples(n_examples=3, seed=None):
    r = np.random.default_rng(seed)
    picks = r.choice(len(te), n_examples, replace=False)
    for q in picks:
        img_id = ids[te[q]]
        print("=" * 72)
        print(f"image {img_id}  -  true caption:")
        print(f'   "{caps[img_id][0]}"')
        if _CAN_SHOW:
            try:
                raw = urllib.request.urlopen(url[img_id], timeout=10).read()
                im = PILImage.open(BytesIO(raw)); im.thumbnail((360, 360))
                display(im)
            except Exception as e:
                print(f"   (image fetch failed: {type(e).__name__} - "
                      f"{url[img_id]})")
        for enc in donors:
            P = l2n((SPACES[enc][te[q:q+1]] @ TO_HUB[enc]) @ HEAD)
            order = np.argsort(-(P @ GAL.T).ravel())[:3]
            hit = "CORRECT @1" if order[0] == q else "          "
            tag = "trained here" if enc == TRAIN_ON else "NEVER SEEN"
            print(f"\n   {enc}  ({tag})   {hit}")
            for rk, g in enumerate(order, 1):
                mark = " <-- true" if g == q else ""
                print(f"      {rk}. {caps[ids[te[g]]][0][:70]}{mark}")
    print("=" * 72)
    print("one head, trained on img_small only - the other encoders were")
    print("never seen by it, in any form, at any point.")

show_examples(3, seed=7)

## The control, live

Replace the fitted hub map with a random one of the same shape. If the
demo above worked because the head is somehow doing the job unaided,
this will work too. It will not.

In [ ]:
def show_control(seed=7):
    r = np.random.default_rng(seed)
    q = int(r.integers(len(te)))
    enc = donors[-1]
    R = np.random.default_rng(99).standard_normal(TO_HUB[enc].shape) / \
        np.sqrt(SPACES[enc].shape[1])
    img_id = ids[te[q]]
    print(f'true caption: "{caps[img_id][0]}"\n')
    P = l2n((SPACES[enc][te[q:q+1]] @ R) @ HEAD)
    order = np.argsort(-(P @ GAL.T).ravel())[:3]
    print(f"{enc} through a RANDOM map - top 3 of 1000:")
    for rk, g in enumerate(order, 1):
        print(f"   {rk}. {caps[ids[te[g]]][0][:70]}")
    print("\nunrelated captions: the transfer above is carried by the")
    print("fitted hub geometry, not by the head alone.")

show_control()

## Live: turn a fresh photo or a typed caption into the shared hub

Everything above reads cached vectors. This cell does the real forward
pass — it loads DINOv2 and bge-m3 on the GPU, encodes an image URL or a
line of text you supply, projects it through the hub, and retrieves
against the same gallery. This is the only part of the demo that needs
CUDA, because it is the only part that runs a neural network rather than
a linear solve.

## Gallery size — how much the demo has to choose from

Both directions retrieve from a fixed pool, and **the size of that pool
is the difficulty of the task**. Measured on this project's own adapter,
same system and same queries, only the gallery changing:

| gallery size | R@1 |
|---|---|
| 10 candidates | 0.967 |
| 100 candidates | 0.852 |
| **1,000 candidates** | **0.557** |

Every number reported in the project uses 1,000. That is what makes
Experiment B, E1, C1.1 and G1 comparable to one another.

For a live demo, 1,000 is unnecessarily restrictive: an uploaded photo
can only be described by a caption that happens to be in the pool. Set
`DEMO_ALL_ROWS = True` to search everything cached, which gives roughly
nine times the coverage.

This is not cheating in either direction — a bigger pool is a *harder*
task, so a score measured here would come out **lower**. It is simply a
different task from the reported protocol, so use the demo to show the
system working, and quote R@1 only from the 1,000-item runs.

In [ ]:
DEMO_ALL_ROWS = True     # False = the strict 1,000-item eval gallery

if DEMO_ALL_ROWS:
    _rows = np.arange(len(SPACES["txt_bge"]))
else:
    _rows = te

GALLERY_IDS      = [ids[i] for i in _rows]
GALLERY_ROWS_IMG = SPACES["img_base" if "img_base" in SPACES
                          else [k for k in SPACES
                                if k.startswith("img_")][0]][_rows]
GAL_DEMO = l2n(SPACES["txt_bge"][_rows])     # caption bank for image->text

print(f"demo gallery: {len(_rows)} items "
      f"({'all cached rows' if DEMO_ALL_ROWS else 'held-out eval only'})")
print()
print("WHY THIS MATTERS. A retrieval score means nothing without its")
print("gallery size, because the size IS the difficulty. Measured on this")
print("project's own adapter, same system and same queries:")
print("     10 candidates -> R@1 0.967")
print("    100 candidates -> R@1 0.852")
print("  1,000 candidates -> R@1 0.557")
print()
print("Every number reported in the project uses 1,000, which is what")
print("makes them comparable to each other. This demo searches a bigger")
print("pool so an uploaded photo has more captions available to match -")
print("that is a HARDER task, so a score measured here would come out")
print("LOWER, not higher. Use the demo to show the system working; quote")
print("R@1 only from the 1,000-item protocol.")

In [ ]:
import torch, urllib.request
from io import BytesIO

_LIVE = None
def _load_live():
    global _LIVE
    if _LIVE is not None:
        return _LIVE
    from transformers import (AutoModel, AutoImageProcessor,
                              AutoTokenizer)
    dev = ("cuda" if torch.cuda.is_available()
           else "mps" if torch.backends.mps.is_available() else "cpu")
    print(f"loading DINOv2-base + bge-m3 on {dev} (first call only)...")
    vp = AutoImageProcessor.from_pretrained("facebook/dinov2-base")
    vm = AutoModel.from_pretrained("facebook/dinov2-base").to(dev).eval()
    tk = AutoTokenizer.from_pretrained("BAAI/bge-m3")
    tm = AutoModel.from_pretrained("BAAI/bge-m3").to(dev).eval()
    _LIVE = (dev, vp, vm, tk, tm)
    return _LIVE

def live_image(source):
    """Encode a real photo through DINOv2, then retrieve captions through
    the hub - the SAME img_base -> hub map used in the measured result.
    `source` may be an http(s) URL, a local file path, or a PIL image."""
    from PIL import Image as PILImage
    dev, vp, vm, tk, tm = _load_live()
    if isinstance(source, PILImage.Image):
        im = source.convert("RGB")
    elif isinstance(source, str) and source.startswith(("http://", "https://")):
        raw = urllib.request.urlopen(source, timeout=15).read()
        im = PILImage.open(BytesIO(raw)).convert("RGB")
    else:
        im = PILImage.open(source).convert("RGB")
    with torch.no_grad():
        b = vp(images=im, return_tensors="pt").to(dev)
        o = vm(**b).last_hidden_state
        v = torch.cat([o[:, 0], o[:, 1:].mean(1)], -1)   # cls+patch
    vec = v.float().cpu().numpy().astype(np.float64)   # [1, 1536]
    q = l2n((vec @ TO_HUB["img_base"]) @ HEAD)          # hub -> bge space
    order = np.argsort(-(q @ GAL_DEMO.T).ravel())[:5]
    try:
        from IPython.display import display; im.thumbnail((300,300)); display(im)
    except Exception: pass
    print(f"retrieved captions from {len(GALLERY_IDS)} candidates "
          f"(image -> hub -> head -> bge space):")
    for r, g in enumerate(order, 1):
        print(f"   {r}. {caps[GALLERY_IDS[g]][0][:72]}")

def live_text(text, top=5, show=True):
    """TEXT -> IMAGES. Encode a query with bge-m3, enter the SHARED HUB
    through bge's own entry map, and rank IMAGES by their hub
    coordinates. This is the mirror of live_image(): the hub is
    symmetric, so either modality can be the query.

    Note what is NOT used here: the caption head. That head maps hub
    coordinates INTO bge space and is only needed when the answer must
    be a caption. Matching image to text inside the hub needs neither.
    """
    from PIL import Image as PILImage
    dev, vp, vm, tk, tm = _load_live()
    with torch.no_grad():
        b = tk(text, return_tensors="pt", truncation=True,
               max_length=64).to(dev)
        h = tm(**b).last_hidden_state
        m = b["attention_mask"].unsqueeze(-1)
        v = (h * m).sum(1) / m.sum(1).clamp(min=1)
    qt = v.float().cpu().numpy().astype(np.float64)

    q_hub = l2n(qt @ TO_HUB["txt_bge"])              # text -> hub
    img_key = "img_base" if "img_base" in SPACES else \
              [k for k in SPACES if k.startswith("img_")][0]
    G_hub = l2n(GALLERY_ROWS_IMG @ TO_HUB[img_key])  # images -> hub
    order = np.argsort(-(q_hub @ G_hub.T).ravel())[:top]

    print(f'text query: "{text}"')
    print(f'searched {len(GALLERY_ROWS_IMG)} images through the hub '
          f'({img_key} side)\n')
    for r, gi in enumerate(order, 1):
        img_id = GALLERY_IDS[gi]
        print(f"   {r}. image {img_id} — {caps[img_id][0][:66]}")
        if show and _CAN_SHOW:
            try:
                raw = urllib.request.urlopen(url[img_id], timeout=10).read()
                im = PILImage.open(BytesIO(raw)); im.thumbnail((220, 220))
                display(im)
            except Exception:
                pass

# Feed a photo in one of three ways:
#   1) a URL (default below - just run the cell)
#   2) upload from your computer:  im = upload_image(); live_image(im)
#   3) a local path on disk:       live_image("/content/my.jpg")
def upload_image():
    """Colab file picker -> PIL image ready for live_image()."""
    from PIL import Image as PILImage
    try:
        from google.colab import files
    except ImportError:
        raise RuntimeError("upload_image() is Colab-only; on a local "
                           "kernel pass a file path to live_image()")
    up = files.upload()
    assert up, "no file chosen"
    name = next(iter(up))
    return PILImage.open(BytesIO(up[name])).convert("RGB")

IMAGE_URL = ("https://media.istockphoto.com/id/1500285927/photo/"
             "young-woman-a-university-student-studying-online.jpg"
             "?s=1024x1024&w=is&k=20&c="
             "CVhpekieDK_UB8vtEDw-dKKGWzDpsxcQt-XEQIkgm3Y=")

live_image(IMAGE_URL)              # set your own URL here, or:
# live_image(upload_image())       # uncomment to upload from your computer
print()
live_text("a group of people riding horses on a beach")

## Persist the hub — so the demo starts instantly

Everything above is rebuilt from the caches in seconds, which is fine at
a desk and awkward on stage. This cell writes the whole assembled system
to one file and reloads it without touching the caches at all.

What goes in, and what does NOT:

- **the hub basis** — a whitened PCA of the five spaces concatenated.
  Built from geometry alone: no captions, no image-text pairing, no
  objective. Shuffling which caption belongs to which image would not
  change it.
- **one entry map per encoder** — closed-form ridge, encoder width to 512.
- **the head** — 512 to bge caption space, fitted on DINOv2-small's hub
  coordinates ONLY. This is the single place where image-caption
  correspondence is used.
- **the caption gallery and its ids** — so retrieval works offline.

Encoder weights are NOT included: the file assumes you already have (or
will fetch) DINOv2 and bge-m3 for fresh inputs. Cached-vector demos need
nothing further.

In [ ]:
HUB_FILE = DATA_DIR / f"hub_{HUB_DIM}d_{len(SPACES)}spaces.npz"

def save_hub(path=HUB_FILE, half=True):
    """Write the assembled system: basis, entry maps, head, gallery."""
    dt = np.float16 if half else np.float32
    blob = {
        "hub_dim":     np.array([HUB_DIM]),
        # dir() inside a function lists LOCAL names, so it can never see
        # a notebook-level variable - use globals() for the fallback.
        "hub_mode":    np.array([globals().get("HUB_MODE", "balanced")]),
        "space_names": np.array(list(SPACES)),
        "trained_on":  np.array([TRAIN_ON]),
        "basis":       BASIS.astype(dt),
        "mu":          _mu.astype(np.float32),
        "head":        HEAD.astype(dt),
        "gallery":     GAL.astype(dt),
        "gallery_ids": np.array([ids[te[i]] for i in range(len(te))]),
        # per-space standardisation constants, needed to enter the hub
        **{f"map__{k}":  TO_HUB[k].astype(dt) for k in SPACES},
        **{f"mean__{k}": SPACES[k][tr].mean(0).astype(np.float32)
           for k in SPACES},
        **{f"scale__{k}": np.array([SPACES[k][tr].std(0).mean()],
                                   dtype=np.float32) for k in SPACES},
    }
    np.savez_compressed(str(path), **blob)
    mb = Path(path).stat().st_size / 1e6
    print(f"saved {Path(path).name}  ({mb:.1f} MB, "
          f"{'fp16' if half else 'fp32'})")
    print("contains: hub basis + one entry map per encoder + the head +")
    print("the caption gallery. NOT the encoder weights.")

def load_hub(path=HUB_FILE):
    """Reload without touching the caches. Returns a dict of arrays."""
    d = np.load(str(path), allow_pickle=False)
    out = {
        "hub_dim": int(d["hub_dim"][0]),
        "names":   [str(x) for x in d["space_names"]],
        "trained_on": str(d["trained_on"][0]),
        "basis":  d["basis"].astype(np.float64),
        "head":   d["head"].astype(np.float64),
        "gallery": d["gallery"].astype(np.float64),
        "gallery_ids": d["gallery_ids"],
        "maps":  {k: d[f"map__{k}"].astype(np.float64)
                  for k in [str(x) for x in d["space_names"]]},
    }
    print(f"loaded {Path(path).name}: {out['hub_dim']}-d hub, "
          f"{len(out['names'])} encoders, head trained on "
          f"{out['trained_on']}")
    return out

save_hub()

# --- verify the reload reproduces the measured numbers exactly ---
_h = load_hub()
_gal = l2n(_h["gallery"])
print(f"\n{'encoder':12s} {'R@1 (from file)':>16s}")
for enc in [k for k in _h["names"] if k.startswith("img_")]:
    P = l2n((SPACES[enc][te] @ _h["maps"][enc]) @ _h["head"])
    r = float(((P @ _gal.T).argmax(1) == np.arange(len(te))).mean())
    tag = "  <- trained here" if enc == _h["trained_on"] else "  <- never seen"
    print(f"{enc:12s} {r:16.3f}{tag}")
print("\nthese should match the table above; fp16 rounding may move the")
print("third decimal. If they differ more than that, the file is stale -")
print("re-run save_hub() after any change to HUB_DIM, ALPHA or the maps.")